## Quantum Sand Project Template Scapy 001

### Import dependencies

We will import `scapy` and `pandas`.

In [1]:
from scapy.all import rdpcap, IP, hexdump, raw
import pandas as pd

### Running Quantum Sand Rails locally and capturing packets using tcpdump

If you are just looking at this notebook statically in a browser or VS Code you can just read this notebook without running tcpdump or the Python code.

Having the Quantum Sand Rails; Ruby-on-Rails PostGIS API; running locally is necessary for running the tcpdump command  and the Python code later in this notebook.

The following tcpdump command has been tested on macOS; The Windows and Linux commands will be added here in the future.

We will use the following flags;

* `-i` the interface, the local network interface and the port 3000.
* `-w` write the raw packets to a file, this is `"tcpdump_rails_api_localhost.pcap"`.

You can read more about these options within the manual for tcpdump using `man tcpdump`.

To exit `man tcpdump` you can type the letter `q`.

The following command was used to generate the existing `"tcpdump_rails_api_localhost.pcap"` within the `quantumsand-project-templates-resources` directory.

```sh
cd quantumsand-project-templates
sudo tcpdump -i lo0 port 3000 -w quantumsand-project-templates-resources/tcpdump_rails_api_localhost.pcap
```

Whilst we leave tcpdump running in the terminal, open a browser like Chrome.

Visit the `geospatial_traces` endpoint within Quantum Sand Rails using this localhost url;

`http://127.0.0.1:3000/geospatial_traces`

Just by visiting the localhost Quantum Sand Rails app, we should have collected some packets with the running tcpdump.

We can now terminate tcpdump. 

On macOS we can use `Control + c` to terminate `tcpdump`.

We can now specify the `tcpdump_rails_api_localhost` file as `"tcpdump_rails_api_localhost.pcap"`.

Using `rdpcap()` from Scapy with the argument `filename` as `tcpdump_rails_api_localhost` we can return a `PacketList` into `packets`.

Using a Python for loop, we can iterate over `packets`.

The Python `if` statement evaluates a condition. 

We can check if each of the packets `haslayer()` of `IP` from Scapy. 

We can then use Python `append()` to add each packet to the Python List `packet_data = []`.

For each of the packets we can add the following information;

* `.src` the source IP address
* `.dst` the destination IP address
* `.len` the packet length in bytes
* `.payload` the actual, usable data being transported in a network packet.

We can then put `packet_data` into `pandas_dataframe`.

The line `pandas_dataframe` outputs a nicely formatted table.

In [2]:
tcpdump_rails_api_localhost = "../quantumsand-project-templates-resources/tcpdump_rails_api_localhost.pcap"
packets = rdpcap(tcpdump_rails_api_localhost)

packet_data = []
for packet in packets:
    if packet.haslayer(IP):
        packet_data.append({
            'src_ip': packet[IP].src,
            'dst_ip': packet[IP].dst,
            'length': packet[IP].len,
            'payload': packet[IP].payload
        })

pandas_dataframe = pd.DataFrame(packet_data)
pandas_dataframe

,src_ip,dst_ip,length,payload
0,127.0.0.1,127.0.0.1,52,[[[TCP 127.0.0.1:62021 > 127.0.0.1:remoteware_...
1,127.0.0.1,127.0.0.1,52,[[[TCP 127.0.0.1:remoteware_cl > 127.0.0.1:620...
2,127.0.0.1,127.0.0.1,828,[[[TCP 127.0.0.1:62019 > 127.0.0.1:remoteware_...
3,127.0.0.1,127.0.0.1,52,[[[TCP 127.0.0.1:remoteware_cl > 127.0.0.1:620...
4,127.0.0.1,127.0.0.1,64,[[[TCP 127.0.0.1:62023 > 127.0.0.1:remoteware_...
5,127.0.0.1,127.0.0.1,64,[[[TCP 127.0.0.1:remoteware_cl > 127.0.0.1:620...
6,127.0.0.1,127.0.0.1,52,[[[TCP 127.0.0.1:62023 > 127.0.0.1:remoteware_...
7,127.0.0.1,127.0.0.1,52,[[[TCP 127.0.0.1:remoteware_cl > 127.0.0.1:620...
8,127.0.0.1,127.0.0.1,612,[[[TCP 127.0.0.1:remoteware_cl > 127.0.0.1:620...
9,127.0.0.1,127.0.0.1,52,[[[TCP 127.0.0.1:62019 > 127.0.0.1:remoteware_...


We can specify the packet we want to look at using `inspect_packet_count`.

In the above table, we can see that the packet with number `2` is `828` bytes in length.

The syntax to access the packet is `packets[inspect_packet_count]`.

Using `hexdump()` from Scapy allows us to look at the raw contents in human-readable hexadecimal and ASCII format.

In [3]:
inspect_packet_count = 2
hexdump(packets[inspect_packet_count])

0000  02 00 00 00 45 00 03 3C 00 00 40 00 40 06 00 00  ....E..<..@.@...
0010  7F 00 00 01 7F 00 00 01 F2 43 0B B8 66 4D 3B C6  .........C..fM;.
0020  14 3D 20 BA 80 18 18 B4 01 31 00 00 01 01 08 0A  .= ......1......
0030  56 EC F1 B6 47 2D 07 52 47 45 54 20 2F 67 65 6F  V...G-.RGET /geo
0040  73 70 61 74 69 61 6C 5F 74 72 61 63 65 73 20 48  spatial_traces H
0050  54 54 50 2F 31 2E 31 0D 0A 48 6F 73 74 3A 20 31  TTP/1.1..Host: 1
0060  32 37 2E 30 2E 30 2E 31 3A 33 30 30 30 0D 0A 43  27.0.0.1:3000..C
0070  6F 6E 6E 65 63 74 69 6F 6E 3A 20 6B 65 65 70 2D  onnection: keep-
0080  61 6C 69 76 65 0D 0A 43 61 63 68 65 2D 43 6F 6E  alive..Cache-Con
0090  74 72 6F 6C 3A 20 6D 61 78 2D 61 67 65 3D 30 0D  trol: max-age=0.
00a0  0A 73 65 63 2D 63 68 2D 75 61 3A 20 22 47 6F 6F  .sec-ch-ua: "Goo
00b0  67 6C 65 20 43 68 72 6F 6D 65 22 3B 76 3D 22 31  gle Chrome";v="1
00c0  34 39 22 2C 20 22 43 68 72 6F 6D 69 75 6D 22 3B  49", "Chromium";
00d0  76 3D 22 31 34 39 22 2C 20 22 4E 6F 74 29 41 3B  v="149", 

Using `raw()` from Scapy allows us to observe the raw packet.

In [4]:
raw(packets[inspect_packet_count])

b'\x02\x00\x00\x00E\x00\x03<\x00\x00@\x00@\x06\x00\x00\x7f\x00\x00\x01\x7f\x00\x00\x01\xf2C\x0b\xb8fM;\xc6\x14= \xba\x80\x18\x18\xb4\x011\x00\x00\x01\x01\x08\nV\xec\xf1\xb6G-\x07RGET /geospatial_traces HTTP/1.1\r\nHost: 127.0.0.1:3000\r\nConnection: keep-alive\r\nCache-Control: max-age=0\r\nsec-ch-ua: "Google Chrome";v="149", "Chromium";v="149", "Not)A;Brand";v="24"\r\nsec-ch-ua-mobile: ?0\r\nsec-ch-ua-platform: "macOS"\r\nUpgrade-Insecure-Requests: 1\r\nUser-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36\r\nAccept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7\r\nSec-Fetch-Site: none\r\nSec-Fetch-Mode: navigate\r\nSec-Fetch-User: ?1\r\nSec-Fetch-Dest: document\r\nAccept-Encoding: gzip, deflate, br, zstd\r\nAccept-Language: en-GB,en-US;q=0.9,en;q=0.8\r\nIf-None-Match: W/"15439268bf60d93676e4f26d7fe3daf6"\r\n\r\n'

Using `show()` will provide lots of detailed information about this packet.

In [5]:
packets[inspect_packet_count].show()

###[ Loopback ]###
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 828
     id        = 0
     flags     = DF
     frag      = 0
     ttl       = 64
     proto     = tcp
     chksum    = 0x0
     src       = 127.0.0.1
     dst       = 127.0.0.1
     \options   \
###[ TCP ]###
        sport     = 62019
        dport     = remoteware_cl
        seq       = 1716337606
        ack       = 339550394
        dataofs   = 8
        reserved  = 0
        flags     = PA
        window    = 6324
        chksum    = 0x131
        urgptr    = 0
        options   = [('NOP', None), ('NOP', None), ('Timestamp', (1458368950, 1194133330))]
###[ Raw ]###
           load      = b'GET /geospatial_traces HTTP/1.1\r\nHost: 127.0.0.1:3000\r\nConnection: keep-alive\r\nCache-Control: max-age=0\r\nsec-ch-ua: "Google Chrome";v="149", "Chromium";v="149", "Not)A;Brand";v="24"\r\nsec-ch-ua-mobile: ?0\r\nsec-ch-ua-platform: "macOS"\r\nUpgrade-Insecure-Reques

Finally, using `json()` returns the packet as JSON.

In [6]:
packets[inspect_packet_count].json()

'{"type": 2, "payload": {"version": 4, "ihl": 5, "tos": 0, "len": 828, "id": 0, "flags": 2, "frag": 0, "ttl": 64, "proto": 6, "chksum": 0, "src": "127.0.0.1", "dst": "127.0.0.1", "payload": {"sport": 62019, "dport": 3000, "seq": 1716337606, "ack": 339550394, "dataofs": 8, "reserved": 0, "flags": 24, "window": 6324, "chksum": 305, "urgptr": 0, "options": ["(\'NOP\', None)", "(\'NOP\', None)", "(\'Timestamp\', (1458368950, 1194133330))"], "payload": {"load": "GET /geospatial_traces HTTP/1.1\\r\\nHost: 127.0.0.1:3000\\r\\nConnection: keep-alive\\r\\nCache-Control: max-age=0\\r\\nsec-ch-ua: \\"Google Chrome\\";v=\\"149\\", \\"Chromium\\";v=\\"149\\", \\"Not)A;Brand\\";v=\\"24\\"\\r\\nsec-ch-ua-mobile: ?0\\r\\nsec-ch-ua-platform: \\"macOS\\"\\r\\nUpgrade-Insecure-Requests: 1\\r\\nUser-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36\\r\\nAccept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/

An example scenario for monitoring network packets is to train an AI model for cybersecurity, which can detect abnormal network traffic.

More to follow...